# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_02 — Cox-Ross-Rubinstein Binomial Lattice

### Research question

How can the Cox-Ross-Rubinstein binomial lattice price European and American options, approximate Black-Scholes-Merton prices, and expose hedge ratios through the structure of a recombining tree?

This notebook extends `02_01_discrete_martingale_pricing_shreve`.

The previous notebook established the discrete-time no-arbitrage logic:

$$
q = \frac{1+r-d}{u-d}
$$

This notebook turns that logic into a full numerical pricing engine using the Cox-Ross-Rubinstein parameterisation:

$$
u = e^{\sigma\sqrt{\Delta t}},
\qquad
d = e^{-\sigma\sqrt{\Delta t}} = \frac{1}{u}
$$

It covers:

1. CRR tree construction;
2. European option pricing;
3. convergence to Black-Scholes-Merton;
4. American option pricing by early-exercise comparison;
5. early-exercise boundary extraction;
6. tree-based Greeks;
7. validation checks;
8. limitations and research-frontier extensions.

The key idea is:

> The CRR lattice is a discrete martingale pricing engine that approximates continuous-time diffusion pricing while preserving no-arbitrage through risk-neutral probabilities.

## 1. CRR model intuition

The Cox-Ross-Rubinstein model approximates geometric Brownian motion:

$$
dS_t = rS_tdt + \sigma S_tdW_t
$$

over a finite grid.

Divide maturity $T$ into $N$ steps:

$$
\Delta t = \frac{T}{N}
$$

At each step, the stock moves up or down:

$$
S_{n+1} =
\begin{cases}
u S_n, & \text{up move} \\
d S_n, & \text{down move}
\end{cases}
$$

The CRR choice is:

$$
u = e^{\sigma\sqrt{\Delta t}},
\qquad
d = e^{-\sigma\sqrt{\Delta t}}
$$

so:

$$
ud = 1
$$

This makes the tree recombining:

$$
S_0ud = S_0du
$$

## 2. Risk-neutral probability

With continuously compounded risk-free rate $r$, the one-step risk-free growth factor is:

$$
e^{r\Delta t}
$$

With continuous dividend yield $q_{div}$, the risk-neutral probability is:

$$
p^* = \frac{e^{(r-q_{div})\Delta t}-d}{u-d}
$$

The no-arbitrage condition is:

$$
d < e^{(r-q_{div})\Delta t} < u
$$

which ensures:

$$
0 < p^* < 1
$$

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class CRRModel:
    """Cox-Ross-Rubinstein binomial lattice parameters."""
    s0: float
    strike: float
    maturity: float
    risk_free_rate: float
    volatility: float
    n_steps: int
    dividend_yield: float = 0.0

    @property
    def dt(self) -> float:
        return self.maturity / self.n_steps

    @property
    def up(self) -> float:
        return exp(self.volatility * sqrt(self.dt))

    @property
    def down(self) -> float:
        return exp(-self.volatility * sqrt(self.dt))

    @property
    def discount(self) -> float:
        return exp(-self.risk_free_rate * self.dt)

    @property
    def forward_growth(self) -> float:
        return exp((self.risk_free_rate - self.dividend_yield) * self.dt)

    @property
    def risk_neutral_prob(self) -> float:
        return (self.forward_growth - self.down) / (self.up - self.down)

    def validate(self) -> None:
        if self.s0 <= 0:
            raise ValueError("s0 must be positive.")
        if self.strike <= 0:
            raise ValueError("strike must be positive.")
        if self.maturity <= 0:
            raise ValueError("maturity must be positive.")
        if self.volatility <= 0:
            raise ValueError("volatility must be positive.")
        if self.n_steps < 1:
            raise ValueError("n_steps must be at least 1.")
        if not (self.down < self.forward_growth < self.up):
            raise ValueError("No-arbitrage condition failed: need d < exp((r-q)dt) < u.")
        if not (0.0 < self.risk_neutral_prob < 1.0):
            raise ValueError("Risk-neutral probability must lie in (0, 1).")


base_model = CRRModel(
    s0=100.0,
    strike=100.0,
    maturity=1.0,
    risk_free_rate=0.05,
    volatility=0.20,
    n_steps=3,
    dividend_yield=0.0,
)

base_model.validate()

pd.Series({
    "dt": base_model.dt,
    "u": base_model.up,
    "d": base_model.down,
    "discount": base_model.discount,
    "risk_neutral_prob": base_model.risk_neutral_prob,
})

## 3. Building the stock-price lattice

At time step $n$, after $j$ up moves, the stock price is:

$$
S_{n,j} = S_0u^jd^{n-j}
$$

where:

$$
j = 0,1,\dots,n
$$

In [ ]:
def build_crr_stock_tree(model: CRRModel) -> list[np.ndarray]:
    """Build a recombining CRR stock-price tree."""
    model.validate()
    tree = []
    for n in range(model.n_steps + 1):
        prices = np.array([
            model.s0 * (model.up ** j) * (model.down ** (n - j))
            for j in range(n + 1)
        ], dtype=float)
        tree.append(prices)
    return tree


def tree_to_dataframe(tree: list[np.ndarray], value_name: str) -> pd.DataFrame:
    """Convert a recombining tree to a long DataFrame."""
    rows = []
    for n, values in enumerate(tree):
        for j, value in enumerate(values):
            rows.append({"time_step": n, "up_moves": j, value_name: float(value)})
    return pd.DataFrame(rows)


stock_tree = build_crr_stock_tree(base_model)
tree_to_dataframe(stock_tree, "stock_price")

## 4. European option pricing by backward induction

At maturity:

$$
V_{N,j}=g(S_{N,j})
$$

Then move backward:

$$
V_{n,j}=e^{-r\Delta t}\left[p^*V_{n+1,j+1}+(1-p^*)V_{n+1,j}\right]
$$

This is dynamic programming on a risk-neutral tree.

In [ ]:
def option_payoff(stock_price: np.ndarray | float, strike: float, option_type: str) -> np.ndarray:
    """Vanilla call or put payoff."""
    stock_price = np.asarray(stock_price, dtype=float)
    if option_type == "call":
        return np.maximum(stock_price - strike, 0.0)
    if option_type == "put":
        return np.maximum(strike - stock_price, 0.0)
    raise ValueError("option_type must be 'call' or 'put'.")


def price_crr_european_tree(model: CRRModel, option_type: str = "call") -> dict:
    """Price a European option using CRR backward induction."""
    model.validate()
    stock_tree = build_crr_stock_tree(model)
    value_tree = [None] * (model.n_steps + 1)
    value_tree[-1] = option_payoff(stock_tree[-1], model.strike, option_type)

    p = model.risk_neutral_prob
    disc = model.discount

    for n in range(model.n_steps - 1, -1, -1):
        next_values = value_tree[n + 1]
        value_tree[n] = disc * (p * next_values[1:] + (1.0 - p) * next_values[:-1])

    return {
        "price": float(value_tree[0][0]),
        "stock_tree": stock_tree,
        "value_tree": value_tree,
        "option_type": option_type,
        "model": model,
    }


call_3 = price_crr_european_tree(base_model, "call")
put_3 = price_crr_european_tree(base_model, "put")

pd.Series({"three_step_call": call_3["price"], "three_step_put": put_3["price"]})

## 5. Terminal distribution validation

For a European option, we can also price directly from the terminal binomial distribution:

$$
V_0=e^{-rT}\sum_{j=0}^{N}\binom{N}{j}(p^*)^j(1-p^*)^{N-j}g(S_{N,j})
$$

This is useful for checking the backward-induction engine.

In [ ]:
def binomial_probabilities(n_steps: int, p: float) -> np.ndarray:
    """Compute binomial probabilities using a stable recurrence."""
    probs = np.empty(n_steps + 1, dtype=float)
    probs[0] = (1.0 - p) ** n_steps
    for j in range(1, n_steps + 1):
        probs[j] = probs[j - 1] * (n_steps - j + 1) / j * p / (1.0 - p)
    return probs


def price_crr_european_terminal_distribution(model: CRRModel, option_type: str) -> float:
    """Price a European option using terminal binomial probabilities."""
    model.validate()
    j = np.arange(model.n_steps + 1)
    terminal_stock = model.s0 * (model.up ** j) * (model.down ** (model.n_steps - j))
    payoff = option_payoff(terminal_stock, model.strike, option_type)
    probs = binomial_probabilities(model.n_steps, model.risk_neutral_prob)
    return float(exp(-model.risk_free_rate * model.maturity) * np.sum(probs * payoff))


pd.Series({
    "tree_call": call_3["price"],
    "terminal_distribution_call": price_crr_european_terminal_distribution(base_model, "call"),
    "absolute_difference": abs(call_3["price"] - price_crr_european_terminal_distribution(base_model, "call")),
})

## 6. Black-Scholes-Merton reference price

The CRR tree should converge toward Black-Scholes-Merton for European options as $N\to\infty$.

With continuous dividend yield $q_{div}$:

$$
C=S_0e^{-q_{div}T}N(d_1)-Ke^{-rT}N(d_2)
$$

$$
P=Ke^{-rT}N(-d_2)-S_0e^{-q_{div}T}N(-d_1)
$$

where:

$$
d_1=\frac{\log(S_0/K)+(r-q_{div}+\frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}},
\qquad
d_2=d_1-\sigma\sqrt{T}
$$

In [ ]:
def normal_cdf(x: float) -> float:
    """Standard normal CDF."""
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


def black_scholes_merton_price(
    s0: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    volatility: float,
    option_type: str = "call",
    dividend_yield: float = 0.0,
) -> float:
    """Black-Scholes-Merton European option price."""
    if maturity <= 0:
        return float(option_payoff(s0, strike, option_type))

    d1 = (
        log(s0 / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility**2) * maturity
    ) / (volatility * sqrt(maturity))
    d2 = d1 - volatility * sqrt(maturity)

    if option_type == "call":
        return s0 * exp(-dividend_yield * maturity) * normal_cdf(d1) - strike * exp(-risk_free_rate * maturity) * normal_cdf(d2)
    if option_type == "put":
        return strike * exp(-risk_free_rate * maturity) * normal_cdf(-d2) - s0 * exp(-dividend_yield * maturity) * normal_cdf(-d1)
    raise ValueError("option_type must be 'call' or 'put'.")


bsm_call = black_scholes_merton_price(100, 100, 1, 0.05, 0.20, "call")
bsm_put = black_scholes_merton_price(100, 100, 1, 0.05, 0.20, "put")

pd.Series({"bsm_call": bsm_call, "bsm_put": bsm_put})

## 7. Convergence to Black-Scholes-Merton

As the number of steps increases, CRR European prices should approach BSM prices.

Finite-step CRR prices may oscillate around the continuous-time value, especially for small $N$.

In [ ]:
def convergence_table(base_parameters: dict, step_grid: list[int], option_type: str) -> pd.DataFrame:
    """Build a CRR convergence table against BSM."""
    bsm_price = black_scholes_merton_price(
        s0=base_parameters["s0"],
        strike=base_parameters["strike"],
        maturity=base_parameters["maturity"],
        risk_free_rate=base_parameters["risk_free_rate"],
        volatility=base_parameters["volatility"],
        option_type=option_type,
        dividend_yield=base_parameters.get("dividend_yield", 0.0),
    )
    rows = []
    for n_steps in step_grid:
        model = CRRModel(**base_parameters, n_steps=n_steps)
        crr_price = price_crr_european_tree(model, option_type)["price"]
        rows.append({
            "n_steps": n_steps,
            "option_type": option_type,
            "crr_price": crr_price,
            "bsm_price": bsm_price,
            "signed_error": crr_price - bsm_price,
            "absolute_error": abs(crr_price - bsm_price),
        })
    return pd.DataFrame(rows)


base_parameters = {
    "s0": 100.0,
    "strike": 100.0,
    "maturity": 1.0,
    "risk_free_rate": 0.05,
    "volatility": 0.20,
    "dividend_yield": 0.0,
}

step_grid = [1, 2, 3, 5, 10, 25, 50, 100, 250, 500, 1000]
call_convergence = convergence_table(base_parameters, step_grid, "call")
put_convergence = convergence_table(base_parameters, step_grid, "put")
call_convergence

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(call_convergence["n_steps"], call_convergence["crr_price"], marker="o", label="CRR call")
plt.axhline(call_convergence["bsm_price"].iloc[0], linestyle="--", label="BSM call")
plt.xscale("log")
plt.title("CRR European Call Convergence to BSM")
plt.xlabel("Number of time steps")
plt.ylabel("Option price")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(call_convergence["n_steps"], call_convergence["absolute_error"], marker="o", label="Call error")
plt.plot(put_convergence["n_steps"], put_convergence["absolute_error"], marker="o", label="Put error")
plt.xscale("log")
plt.yscale("log")
plt.title("CRR Absolute Error vs BSM")
plt.xlabel("Number of time steps")
plt.ylabel("Absolute error, log scale")
plt.legend()
plt.show()

## 8. Put-call parity validation

European options should satisfy:

$$
C-P=S_0e^{-q_{div}T}-Ke^{-rT}
$$

This is a useful unit test for the pricing engine.

In [ ]:
def put_call_parity_error(call_price: float, put_price: float, model: CRRModel) -> float:
    rhs = model.s0 * exp(-model.dividend_yield * model.maturity) - model.strike * exp(-model.risk_free_rate * model.maturity)
    return (call_price - put_price) - rhs


parity_rows = []
for n_steps in [3, 10, 50, 250, 1000]:
    m = CRRModel(**base_parameters, n_steps=n_steps)
    call = price_crr_european_tree(m, "call")["price"]
    put = price_crr_european_tree(m, "put")["price"]
    parity_rows.append({
        "n_steps": n_steps,
        "call_price": call,
        "put_price": put,
        "parity_error": put_call_parity_error(call, put, m),
    })

parity_df = pd.DataFrame(parity_rows)
parity_df

## 9. American option pricing

For American options, early exercise is allowed.

At each node:

$$
V_{n,j}=\max\left(g(S_{n,j}),e^{-r\Delta t}\left[p^*V_{n+1,j+1}+(1-p^*)V_{n+1,j}\right]\right)
$$

For a non-dividend-paying stock, early exercise of an American call is usually not optimal.

For puts, early exercise can be optimal when the option is sufficiently deep in the money.

In [ ]:
def price_crr_american_tree(model: CRRModel, option_type: str = "put") -> dict:
    """Price an American option using early-exercise backward induction."""
    model.validate()
    stock_tree = build_crr_stock_tree(model)
    value_tree = [None] * (model.n_steps + 1)
    exercise_tree = [None] * (model.n_steps + 1)
    value_tree[-1] = option_payoff(stock_tree[-1], model.strike, option_type)
    exercise_tree[-1] = value_tree[-1] > 0

    p = model.risk_neutral_prob
    disc = model.discount

    for n in range(model.n_steps - 1, -1, -1):
        continuation = disc * (p * value_tree[n + 1][1:] + (1.0 - p) * value_tree[n + 1][:-1])
        exercise = option_payoff(stock_tree[n], model.strike, option_type)
        value_tree[n] = np.maximum(continuation, exercise)
        exercise_tree[n] = exercise > continuation + 1e-12

    return {
        "price": float(value_tree[0][0]),
        "stock_tree": stock_tree,
        "value_tree": value_tree,
        "exercise_tree": exercise_tree,
        "option_type": option_type,
        "model": model,
    }


american_model = CRRModel(**base_parameters, n_steps=300)
american_put = price_crr_american_tree(american_model, "put")
european_put = price_crr_european_tree(american_model, "put")
american_call = price_crr_american_tree(american_model, "call")
european_call = price_crr_european_tree(american_model, "call")

pd.Series({
    "american_put": american_put["price"],
    "european_put": european_put["price"],
    "put_early_exercise_premium": american_put["price"] - european_put["price"],
    "american_call": american_call["price"],
    "european_call": european_call["price"],
    "call_early_exercise_premium": american_call["price"] - european_call["price"],
})

## 10. Early-exercise boundary

For an American put, early exercise is typically optimal when the stock price is sufficiently low.

We extract the highest stock price at each time step where exercise is optimal.

In [ ]:
def extract_early_exercise_boundary(american_result: dict) -> pd.DataFrame:
    """Extract early-exercise boundary from an American option tree."""
    stock_tree = american_result["stock_tree"]
    exercise_tree = american_result["exercise_tree"]
    rows = []
    for n in range(len(exercise_tree) - 1):
        flags = exercise_tree[n]
        prices = stock_tree[n]
        if np.any(flags):
            exercise_prices = prices[flags]
            rows.append({
                "time_step": n,
                "min_exercise_stock": float(np.min(exercise_prices)),
                "max_exercise_stock": float(np.max(exercise_prices)),
                "num_exercise_nodes": int(np.sum(flags)),
            })
        else:
            rows.append({
                "time_step": n,
                "min_exercise_stock": np.nan,
                "max_exercise_stock": np.nan,
                "num_exercise_nodes": 0,
            })
    return pd.DataFrame(rows)


exercise_boundary = extract_early_exercise_boundary(american_put)
exercise_boundary.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(exercise_boundary["time_step"], exercise_boundary["max_exercise_stock"], marker="o", markersize=3)
plt.axhline(american_model.strike, linestyle="--", label="Strike")
plt.title("American Put Early-Exercise Boundary")
plt.xlabel("Time step")
plt.ylabel("Highest stock price where exercise is optimal")
plt.legend()
plt.show()

## 11. Tree-based Greeks

The tree gives finite-difference approximations to Greeks.

Delta near the root:

$$
\Delta \approx \frac{V_u-V_d}{S_u-S_d}
$$

Gamma is approximated from deltas one level lower.

Theta is approximated by comparing the root value with a later central node.

In [ ]:
def crr_tree_greeks(result: dict) -> pd.Series:
    """Estimate Delta, Gamma, and Theta from a CRR tree."""
    model = result["model"]
    stock_tree = result["stock_tree"]
    value_tree = result["value_tree"]

    delta = (value_tree[1][1] - value_tree[1][0]) / (stock_tree[1][1] - stock_tree[1][0])
    gamma = np.nan
    theta = np.nan

    if model.n_steps >= 2:
        delta_down = (value_tree[2][1] - value_tree[2][0]) / (stock_tree[2][1] - stock_tree[2][0])
        delta_up = (value_tree[2][2] - value_tree[2][1]) / (stock_tree[2][2] - stock_tree[2][1])
        stock_mid_down = 0.5 * (stock_tree[2][1] + stock_tree[2][0])
        stock_mid_up = 0.5 * (stock_tree[2][2] + stock_tree[2][1])
        gamma = (delta_up - delta_down) / (stock_mid_up - stock_mid_down)
        theta = (value_tree[2][1] - value_tree[0][0]) / (2 * model.dt)

    return pd.Series({"delta": float(delta), "gamma": float(gamma), "theta": float(theta)})


greek_model = CRRModel(**base_parameters, n_steps=500)
greek_call = price_crr_european_tree(greek_model, "call")
greek_put = price_crr_european_tree(greek_model, "put")

pd.DataFrame({"call": crr_tree_greeks(greek_call), "put": crr_tree_greeks(greek_put)})

## 12. Black-Scholes Delta comparison

For a European call:

$$
\Delta_C=e^{-q_{div}T}N(d_1)
$$

For a European put:

$$
\Delta_P=e^{-q_{div}T}(N(d_1)-1)
$$

In [ ]:
def bsm_delta(s0, strike, maturity, risk_free_rate, volatility, option_type, dividend_yield=0.0) -> float:
    """Black-Scholes-Merton Delta."""
    d1 = (log(s0 / strike) + (risk_free_rate - dividend_yield + 0.5 * volatility**2) * maturity) / (volatility * sqrt(maturity))
    if option_type == "call":
        return exp(-dividend_yield * maturity) * normal_cdf(d1)
    if option_type == "put":
        return exp(-dividend_yield * maturity) * (normal_cdf(d1) - 1.0)
    raise ValueError("option_type must be 'call' or 'put'.")


pd.Series({
    "crr_call_delta": crr_tree_greeks(greek_call)["delta"],
    "bsm_call_delta": bsm_delta(option_type="call", **base_parameters),
    "crr_put_delta": crr_tree_greeks(greek_put)["delta"],
    "bsm_put_delta": bsm_delta(option_type="put", **base_parameters),
})

## 13. Visualising a small lattice

For small trees, node visualisation makes the recursion transparent.

Each node below displays:

```text
S = stock price
V = option value
```

In [ ]:
def plot_lattice_tree(stock_tree: list[np.ndarray], value_tree: list[np.ndarray] | None = None, title: str = "CRR Lattice") -> None:
    """Plot a small recombining lattice."""
    xs, ys, labels = [], [], []
    for n, prices in enumerate(stock_tree):
        for j, price in enumerate(prices):
            xs.append(n)
            ys.append(j - n / 2)
            if value_tree is None:
                labels.append(f"{price:.1f}")
            else:
                labels.append(f"S={price:.1f}\nV={value_tree[n][j]:.2f}")

    plt.figure(figsize=(11, 6))
    plt.scatter(xs, ys)
    for x, y, label in zip(xs, ys, labels):
        plt.text(x, y, label, ha="center", va="bottom", fontsize=8)
    plt.title(title)
    plt.xlabel("Time step")
    plt.ylabel("State index")
    plt.grid(True, alpha=0.3)
    plt.show()


plot_lattice_tree(call_3["stock_tree"], call_3["value_tree"], title="Three-Step CRR Call Lattice")

## 14. Price surface over volatility and moneyness

A good pricing engine should behave sensibly across inputs.

For a call option, price should generally increase as:

- volatility increases;
- moneyness $S_0/K$ increases.

In [ ]:
def price_grid_vol_moneyness(
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    n_steps: int,
    vol_grid: np.ndarray,
    moneyness_grid: np.ndarray,
    option_type: str = "call",
) -> pd.DataFrame:
    rows = []
    for vol in vol_grid:
        for m in moneyness_grid:
            model = CRRModel(
                s0=m * strike,
                strike=strike,
                maturity=maturity,
                risk_free_rate=risk_free_rate,
                volatility=vol,
                n_steps=n_steps,
                dividend_yield=dividend_yield,
            )
            price = price_crr_european_tree(model, option_type)["price"]
            rows.append({"volatility": vol, "moneyness": m, "price": price, "option_type": option_type})
    return pd.DataFrame(rows)


vol_grid = np.linspace(0.10, 0.60, 11)
moneyness_grid = np.linspace(0.75, 1.25, 11)
call_surface = price_grid_vol_moneyness(100, 1.0, 0.05, 0.0, 250, vol_grid, moneyness_grid, "call")
call_surface.head()

In [ ]:
pivot_surface = call_surface.pivot(index="volatility", columns="moneyness", values="price")

plt.figure(figsize=(10, 6))
plt.imshow(
    pivot_surface.values,
    aspect="auto",
    origin="lower",
    extent=[moneyness_grid.min(), moneyness_grid.max(), vol_grid.min(), vol_grid.max()],
)
plt.colorbar(label="Call price")
plt.title("CRR Call Price Surface")
plt.xlabel("Moneyness S0/K")
plt.ylabel("Volatility")
plt.show()

## 15. Validation checklist

A CRR pricing engine should satisfy:

1. risk-neutral probability lies in $(0,1)$;
2. European put-call parity holds;
3. European prices converge toward BSM;
4. American option value is at least European value;
5. American call without dividends approximately equals European call;
6. option values are non-negative;
7. terminal payoffs are correct.

In [ ]:
def run_crr_validation_checks() -> pd.DataFrame:
    """Run basic validation checks for the CRR engine."""
    model = CRRModel(**base_parameters, n_steps=250)
    model.validate()
    call = price_crr_european_tree(model, "call")
    put = price_crr_european_tree(model, "put")
    amer_put = price_crr_american_tree(model, "put")
    amer_call = price_crr_american_tree(model, "call")

    bsm_call_ref = black_scholes_merton_price(option_type="call", **base_parameters)
    parity_error = put_call_parity_error(call["price"], put["price"], model)

    checks = [
        {
            "check": "risk_neutral_probability_in_unit_interval",
            "passed": 0 < model.risk_neutral_prob < 1,
            "value": model.risk_neutral_prob,
        },
        {
            "check": "european_put_call_parity",
            "passed": abs(parity_error) < 1e-8,
            "value": parity_error,
        },
        {
            "check": "crr_close_to_bsm_call",
            "passed": abs(call["price"] - bsm_call_ref) < 0.05,
            "value": call["price"] - bsm_call_ref,
        },
        {
            "check": "american_put_at_least_european_put",
            "passed": amer_put["price"] >= put["price"] - 1e-10,
            "value": amer_put["price"] - put["price"],
        },
        {
            "check": "non_dividend_american_call_equals_european_call",
            "passed": abs(amer_call["price"] - call["price"]) < 1e-8,
            "value": amer_call["price"] - call["price"],
        },
    ]
    return pd.DataFrame(checks)


validation_checks = run_crr_validation_checks()
validation_checks

## 16. Limitations

The CRR lattice is powerful, but simplified.

### 16.1 Constant volatility

The tree assumes one constant volatility parameter. Real markets have volatility smiles, skew, term structure, stochastic volatility, and jumps.

### 16.2 Constant interest rate

The notebook assumes a constant continuously compounded rate.

### 16.3 Recombining structure restricts dynamics

The recombining tree is efficient because $ud=1$, but this restricts possible dynamics.

### 16.4 Discrete approximation error

Finite-step prices can oscillate around the continuous-time value.

### 16.5 Frictionless markets

The model assumes no bid-ask spread, transaction costs, market impact, margin constraints, or funding constraints.

### 16.6 American exercise is model-dependent

Changing volatility, dividends, rates, or constraints can change the exercise boundary.

## 17. Research frontier and current directions

### 17.1 Adaptive and non-uniform trees

Uniform time steps may be inefficient near barriers, dividends, or maturity. Adaptive lattices refine the grid where accuracy matters most.

### 17.2 Trinomial and local-volatility trees

Trinomial trees allow up, middle, and down moves. Local-volatility trees calibrate node-dependent dynamics to observed option smiles.

### 17.3 American pricing beyond trees

American options can also be priced with finite-difference PDE methods, Longstaff-Schwartz Monte Carlo, and dual martingale bounds.

### 17.4 Discrete hedging error

The CRR hedge is exact on the tree. In real markets, hedging is discrete, costly, and exposed to model error.

### 17.5 Robust and model-independent pricing

A single tree assumes one volatility and one dynamic model. Robust pricing asks for price intervals across plausible models.

### 17.6 Hardware-accelerated lattices

Large option books can require pricing across many strikes, maturities, and scenarios. Vectorisation, Numba, C++, JAX, or GPU kernels can accelerate batch lattice pricing.

## 18. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_03_black_scholes_merton_pde**  
   Deriving the continuous-time limit and PDE logic.

2. **02_04_implied_volatility_root_finding**  
   Inverting option prices to recover implied volatility.

3. **02_07_exotic_options_monte_carlo**  
   Pricing path-dependent options where recombining trees are less convenient.

4. **02_08_dynamic_delta_hedging_simulation**  
   Testing hedging error under discrete rebalancing.

5. **02_10_finite_difference_pde_pricer**  
   Solving option-pricing PDEs on grids.

6. **02_11_heston_model_calibration**  
   Moving from constant volatility to stochastic volatility.

## 19. Saving lattice outputs

The notebook saves:

1. convergence tables;
2. European parity checks;
3. American early-exercise boundary;
4. call price surface;
5. validation checks;
6. manifest with model parameters.

In [ ]:
output_dir = Path("data/processed/crr_binomial_lattice_v1")
output_dir.mkdir(parents=True, exist_ok=True)

call_convergence_path = output_dir / "call_convergence_to_bsm.csv"
put_convergence_path = output_dir / "put_convergence_to_bsm.csv"
parity_path = output_dir / "put_call_parity_checks.csv"
exercise_boundary_path = output_dir / "american_put_exercise_boundary.csv"
surface_path = output_dir / "call_price_surface.csv"
validation_path = output_dir / "validation_checks.csv"
tree_path = output_dir / "three_step_call_tree.csv"
manifest_path = output_dir / "manifest.json"

call_convergence.to_csv(call_convergence_path, index=False)
put_convergence.to_csv(put_convergence_path, index=False)
parity_df.to_csv(parity_path, index=False)
exercise_boundary.to_csv(exercise_boundary_path, index=False)
call_surface.to_csv(surface_path, index=False)
validation_checks.to_csv(validation_path, index=False)

three_step_tree_df = tree_to_dataframe(call_3["stock_tree"], "stock_price").merge(
    tree_to_dataframe(call_3["value_tree"], "option_value"),
    on=["time_step", "up_moves"],
    how="left",
)
three_step_tree_df.to_csv(tree_path, index=False)

manifest = {
    "dataset_name": "crr_binomial_lattice_outputs",
    "schema_version": "crr_binomial_lattice_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "base_parameters": base_parameters,
    "bsm_call_reference": float(bsm_call),
    "bsm_put_reference": float(bsm_put),
    "american_put_model": asdict(american_model),
    "american_put_price": float(american_put["price"]),
    "european_put_price_same_grid": float(european_put["price"]),
    "early_exercise_premium": float(american_put["price"] - european_put["price"]),
    "notes": [
        "CRR parameters use u=exp(sigma sqrt(dt)) and d=1/u.",
        "European prices are compared to Black-Scholes-Merton.",
        "American options are priced by max(continuation, exercise).",
        "Outputs are synthetic model outputs, not market prices.",
    ],
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

call_convergence_path, exercise_boundary_path, validation_path, manifest_path

## 20. Summary

This notebook implemented the Cox-Ross-Rubinstein binomial lattice.

It showed:

1. CRR up/down parameters:

$$
u=e^{\sigma\sqrt{\Delta t}},
\qquad
d=e^{-\sigma\sqrt{\Delta t}}
$$

2. risk-neutral probability:

$$
p^*=\frac{e^{(r-q_{div})\Delta t}-d}{u-d}
$$

3. European option pricing by backward induction;
4. validation by terminal distribution pricing;
5. convergence to Black-Scholes-Merton;
6. put-call parity;
7. American option pricing by early exercise;
8. early-exercise boundary extraction;
9. tree-based Greeks;
10. price sensitivity across volatility and moneyness.

The key mathematical takeaway is:

> The CRR lattice is a discrete martingale pricing engine that approximates continuous-time diffusion pricing while preserving no-arbitrage through the risk-neutral probability.

The key financial takeaway is:

> A tree is not only a pricing tool. It also exposes the hedge, exercise boundary, and numerical assumptions behind the price.

## 21. Further reading

### Core references

- Cox, J. C., Ross, S. A., and Rubinstein, M. "Option Pricing: A Simplified Approach."
- Shreve, S. E. *Stochastic Calculus for Finance I: The Binomial Asset Pricing Model.*
- Hull, J. C. *Options, Futures, and Other Derivatives.*
- Björk, T. *Arbitrage Theory in Continuous Time.*

### Extensions

- Trinomial trees.
- Local-volatility trees.
- American option finite-difference methods.
- Longstaff-Schwartz least-squares Monte Carlo.
- Discrete hedging error.
- Martingale optimal transport.
- Robust pricing under model uncertainty.